# Audit dan eksperimen awal ulasan wisata Madura
**Asistensi Hibah Penelitian · Malik Hariyanto**

Unggah `koreksi_dataset.xlsx` melalui panel **Files** di Colab, lalu jalankan sel berurutan. Ubah `DATA_PATH` jika nama atau lokasi file berbeda. Luaran agregat disimpan di folder `hasil_audit_wisata`.

Unit data adalah ulasan, sedangkan label `rekomendasi_cuaca` dan `wisata` pada file ini merupakan atribut destinasi. Eksperimen menguji prediksi kedua atribut dari teks pada destinasi yang ditahan untuk uji. Prediksi tersebut tidak mengukur cuaca saat kunjungan, sentimen, atau relevansi rekomendasi bagi pengguna.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd
import sklearn
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score, hamming_loss
from sklearn.svm import LinearSVC

DATA_PATH = Path('koreksi_dataset.xlsx')
SHEET = 'Data Ulasan'
OUT = Path('hasil_audit_wisata')
SEED = 42
TARGETS = ['rekomendasi_cuaca', 'wisata']
REQUIRED = [
    'id_destinasi', 'destinasi', 'rating', 'date', 'text_original',
    'text_preprocessed', *TARGETS,
]

if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Unggah file dan sesuaikan DATA_PATH: {DATA_PATH}')
data = pd.read_excel(DATA_PATH, sheet_name=SHEET)
missing = sorted(set(REQUIRED) - set(data.columns))
if missing:
    raise ValueError(f'Kolom yang belum tersedia: {missing}')
if data.empty or data[['id_destinasi', 'destinasi', *TARGETS]].isna().any().any():
    raise ValueError('File kosong atau ID, nama destinasi, atau label memiliki nilai kosong.')

OUT.mkdir(exist_ok=True)

## 1 Audit kualitas data
Kolom tanggal diperiksa dalam bentuk aslinya. Frasa relatif seperti *2 tahun lalu* tidak dikonversi menjadi tanggal pasti. Pola karakter rusak adalah indikator untuk peninjauan manual. Teks asli disimpan di file sumber dan tidak disalin ke ekspor agregat.

In [ ]:
def normalize_key(series):
    return (
        series.fillna('').astype(str).str.normalize('NFKC').str.casefold()
        .str.replace(r'[\W_]+', ' ', regex=True).str.strip()
    )

text = normalize_key(data['text_preprocessed'])
original = data['text_original'].fillna('').astype(str)
relative = data['date'].astype(str).str.contains(r'\blalu\b', case=False, regex=True)
target_variation = data.groupby('id_destinasi')[TARGETS].nunique()
profile = {
    'file_sha256': hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    'python': sys.version.split()[0], 'pandas': pd.__version__,
    'scikit_learn': sklearn.__version__, 'seed': SEED,
    'rows': len(data), 'destinations': int(data['id_destinasi'].nunique()),
    'empty_by_column': {k: int(v) for k, v in data[REQUIRED].isna().sum().items()},
    'exact_duplicate_rows': int(data.duplicated(subset=REQUIRED).sum()),
    'relative_date_rows': int(relative.sum()),
    'potential_encoding_error_rows': int(original.str.contains(r'ðŸ|â€¦|Ã©|�', case=False, regex=True).sum()),
    'empty_normalized_text_rows': int(text.eq('').sum()),
    'rating_counts': {str(k): int(v) for k, v in data['rating'].value_counts().sort_index().items()},
    'labels_changing_within_destination': {
        label: int(target_variation[label].gt(1).sum()) for label in TARGETS
    },
}
if target_variation.gt(1).any().any():
    raise ValueError('Ada label yang berubah di dalam satu ID destinasi. Periksa unit label.')
if data.groupby('id_destinasi')['destinasi'].nunique().gt(1).any():
    raise ValueError('Satu ID mengacu pada beberapa nama destinasi. Periksa pemetaan ID.')

counts = pd.concat([
    data.groupby(label, sort=True).agg(
        ulasan=('id_destinasi', 'size'), destinasi=('id_destinasi', 'nunique')
    ).reset_index(names='kelas').assign(target=label)
    for label in TARGETS
], ignore_index=True)[['target', 'kelas', 'ulasan', 'destinasi']]
counts.to_csv(OUT / 'distribusi_label.csv', index=False)
print(json.dumps(profile, indent=2, ensure_ascii=False))
print(counts.to_string(index=False))

## 2 Pemisahan destinasi dan teks berulang
Ambil sekitar 20 persen ID dari tiap jenis wisata, dengan minimal satu ID di tiap bagian jika kelompok memungkinkan. Rasio akhir dihitung dari jumlah destinasi, sehingga rasio jumlah ulasan dapat berbeda. Teks identik dalam satu destinasi dihitung sekali. Teks uji yang juga ada di latih dikeluarkan dari evaluasi agar frasa yang sama tidak muncul di kedua sisi. Penghapusan ini bisa mengubah komposisi data, jadi jumlah yang keluar dicatat. Periksa cakupan kelas sebelum membaca metrik.

In [ ]:
rows = data.assign(teks_model=text).loc[text.ne('')].copy()
work = rows.drop_duplicates(subset=['id_destinasi', 'teks_model'])
destinations = work[['id_destinasi', *TARGETS]].drop_duplicates().sort_values('id_destinasi')
rng = np.random.default_rng(SEED)
test_ids = set()
for _, group in destinations.groupby('wisata', sort=True):
    ids = group['id_destinasi'].to_numpy()
    if len(ids) < 2:
        continue
    test_count = min(len(ids) - 1, max(1, int(round(len(ids) * 0.20))))
    test_ids.update(rng.choice(ids, size=test_count, replace=False))

train = work.loc[~work['id_destinasi'].isin(test_ids)]
test_before_filter = work.loc[work['id_destinasi'].isin(test_ids)]
test = test_before_filter.loc[~test_before_filter['teks_model'].isin(train['teks_model'])]
if train.empty or test.empty:
    raise ValueError('Data latih atau data uji kosong setelah pemisahan dan penghapusan overlap.')
assert not set(train['id_destinasi']) & set(test['id_destinasi'])
assert not set(train['teks_model']) & set(test['teks_model'])
profile.update({
    'duplicates_within_destination_removed': len(rows) - len(work),
    'test_text_overlap_removed': len(test_before_filter) - len(test),
    'train_reviews': len(train), 'test_reviews': len(test),
    'train_destinations': int(train['id_destinasi'].nunique()),
    'test_destinations': int(test['id_destinasi'].nunique()),
    'heldout_destinations_without_evaluable_reviews': int(len(test_ids) - test['id_destinasi'].nunique()),
    'classes_absent_from_test': {
        label: sorted(set(train[label]) - set(test[label])) for label in TARGETS
    },
})
for label in TARGETS:
    if train[label].nunique() < 2 or set(test[label]) - set(train[label]):
        raise ValueError(f'Pembagian data tidak bisa mengevaluasi target {label}.')
profile_path = OUT / 'audit_dataset.json'
profile_path.write_text(json.dumps(profile, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(profile, ensure_ascii=False, indent=2))

## 3 Pembanding untuk dua atribut destinasi
Model mayoritas memberi titik acuan. Model linear memakai TF-IDF yang dipelajari dari teks latih saja. Keduanya diuji pada ulasan yang sama. Untuk hasil per destinasi, prediksi ulasan digabung melalui suara terbanyak. Jika seri, label pertama dalam urutan alfabet dipilih secara deterministik. Macro-F1 memberi bobot yang sama pada tiap kelas yang hadir di data uji. Micro-F1 sama dengan akurasi untuk satu label per baris. Baca jumlah contoh per kelas, khususnya kelas dengan sedikit destinasi.

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), min_df=2, max_features=40_000, sublinear_tf=True,
    token_pattern=r'(?u)\b\w+\b',
)
train_x = vectorizer.fit_transform(train['teks_model'])
test_x = vectorizer.transform(test['teks_model'])

def measure(y_true, y_pred):
    labels = sorted(set(y_true))
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'micro_f1': f1_score(y_true, y_pred, average='micro'),
        'macro_f1': f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0),
        'hamming_loss': hamming_loss(y_true, y_pred),
        'n': len(y_true),
    }

scores, details = [], []
for label in TARGETS:
    classifiers = {
        'mayoritas': DummyClassifier(strategy='most_frequent'),
        'tfidf_linear_svc': LinearSVC(class_weight='balanced', random_state=SEED),
    }
    for model_name, classifier in classifiers.items():
        classifier.fit(train_x, train[label])
        prediction = classifier.predict(test_x)
        grouped = (
            test[['id_destinasi', label]].assign(prediksi=prediction)
            .groupby('id_destinasi', sort=True)
            .agg(aktual=(label, 'first'), prediksi=('prediksi', lambda s: s.mode().iloc[0]))
        )
        for unit, actual, predicted in [
            ('ulasan', test[label], prediction),
            ('destinasi', grouped['aktual'], grouped['prediksi']),
        ]:
            scores.append({'target': label, 'model': model_name, 'unit': unit,
                           **measure(actual, predicted)})
            report = classification_report(actual, predicted, labels=sorted(set(actual)),
                                           output_dict=True, zero_division=0)
            for cls in sorted(set(actual)):
                details.append({'target': label, 'model': model_name, 'unit': unit,
                                'kelas': cls, **report[cls]})
metrics = pd.DataFrame(scores)
metrics.to_csv(OUT / 'metrik_baseline.csv', index=False)
pd.DataFrame(details).to_csv(OUT / 'metrik_per_kelas.csv', index=False)
print(metrics.round(4).to_string(index=False))

## 4 Eksplorasi topik dengan LDA
Model topik memakai sampel data latih dengan maksimum 30 ulasan per destinasi. Satu kosakata dan satu model bersama membuat nomor topik dapat dibandingkan antar kategori kecocokan cuaca. Rerata dihitung pada tingkat destinasi. Topik ini bersifat deskriptif. Untuk membahas perubahan topik menurut waktu kunjungan, diperlukan tanggal absolut dan aturan pelabelan musim yang tervalidasi.

In [ ]:
sample = (
    train.sample(frac=1, random_state=SEED).groupby('id_destinasi', sort=False)
    .head(30).reset_index(drop=True)
)
counts_vectorizer = CountVectorizer(min_df=3, max_features=4000, token_pattern=r'(?u)\b\w\w+\b')
sample_counts = counts_vectorizer.fit_transform(sample['teks_model'])
lda = LatentDirichletAllocation(n_components=6, max_iter=8, learning_method='online',
                                random_state=SEED, n_jobs=1)
weights = lda.fit_transform(sample_counts)
vocabulary = counts_vectorizer.get_feature_names_out()
topic_names = [f'topik_{i + 1}' for i in range(weights.shape[1])]
top_terms = pd.DataFrame({
    'topik': topic_names,
    'kata_utama': [', '.join(vocabulary[np.argsort(row)[-8:][::-1]]) for row in lda.components_],
})
topic_by_destination = (
    pd.DataFrame(weights, columns=topic_names)
    .assign(id_destinasi=sample['id_destinasi'], rekomendasi_cuaca=sample['rekomendasi_cuaca'])
    .groupby(['id_destinasi', 'rekomendasi_cuaca'])[topic_names].mean()
)
topic_by_weather = topic_by_destination.groupby('rekomendasi_cuaca')[topic_names].mean()
top_terms.to_csv(OUT / 'topik_lda.csv', index=False)
topic_by_weather.to_csv(OUT / 'ringkasan_topik_per_label_cuaca.csv')
profile.update({'lda_sample_reviews': len(sample), 'lda_topics': len(topic_names)})
profile_path.write_text(json.dumps(profile, ensure_ascii=False, indent=2), encoding='utf-8')
print(top_terms.to_string(index=False))
print(topic_by_weather.round(4).to_string())

## 5 Rancangan eksperimen setelah label dikonfirmasi
1. **Data dan label.** Dokumentasikan sumber label kecocokan cuaca dan kategori wisata. Pastikan perlakuan untuk destinasi di luar Madura. Verifikasi pra-pemrosesan, termasuk negasi. Untuk sentimen, buat anotasi teks dan peninjauan sebagian label. Jika rating dipakai sebagai label proksi, evaluasi terhadap anotasi manusia.
2. **Klasifikasi.** Latih FastText pada korpus latih, lalu gunakan vektornya sebagai masukan BiLSTM dengan kepala prediksi per aspek yang sah. Bandingkan skenario FastText saja, LDA saja dengan pengklasifikasi vektor topik, dan gabungan BiLSTM–LDA menggunakan partisi yang sama. Sesuaikan parameter di validasi latih, lalu bekukan set uji.
3. **Konteks musim.** Pisahkan kondisi cuaca atau musim dari penanda libur. Jika musim adalah target, jangan memilih model LDA memakai label musim sebenarnya. Dokumentasikan definisi musim, sumber tanggal dan wilayah sebelum menganalisis tren waktu.
4. **Rekomendasi.** Siapkan katalog destinasi, kueri atau preferensi, dan daftar destinasi relevan yang dinilai independen. Uji BGE dan pemeringkatan berbasis konteks dengan Precision@K dan Recall@K pada kueri terpisah.
5. **Struktur repositori.** Simpan eksperimen awal di `notebooks/`. Setelah metode disepakati, pindahkan proses yang dipakai ulang ke `src/` untuk audit, pemisahan data, ekstraksi fitur, pelatihan dan evaluasi. Simpan konfigurasi di `configs/` dan ringkasan yang dapat dibagikan di `reports/`. Catat versi dependensi, seed, hash data dan ID eksperimen.